In [ ]:
import pandas as pd
from pathlib import Path

DATA_DIR = Path(r"C:\Users\SM24519\OneDrive - Cardiff Metropolitan University\Desktop\Data\Papers\Presentations and directions\Reviews\Year 3\final_phase_documents\tis_replica_v3\output")

In [ ]:
# individual systems (each source working independently) vs the TIS bundle (accumulated view)
individual = {}
tis_bundle = {}

for f in DATA_DIR.glob("*.csv"):
    df = pd.read_csv(f)
    if f.stem.startswith("patient_history"):
        tis_bundle[f.stem] = df
    else:
        individual[f.stem] = df

for name, df in {**individual, **tis_bundle}.items():
    print(f"{name}: {df.shape[0]} rows, {df.shape[1]} cols")

## Step 2: unit normalization + time alignment

Individual systems report the same concept under different names/units (India "Total Cholesterol" mg/dl vs UK "Cholesterol" mmol/L — these even carry different LOINC codes, since LOINC is unit-specific). This maps everything to one canonical name/unit and one parsed timestamp so sources can be compared directly.

In [ ]:
from datetime import datetime

# same concept, different name per source -> canonical name (matches the TIS bundle's own wording)
CONCEPT_MAP = {
    "triglycerides": "Triglycerides", "triglyceride": "Triglycerides",
    "total cholesterol": "Total cholesterol", "cholesterol": "Total cholesterol",
    "hdl cholesterol": "HDL cholesterol",
    "non-hdl cholesterol": "Non-HDL cholesterol", "non hdl cholesterol": "Non-HDL cholesterol",
    "ldl cholesterol": "LDL cholesterol",
    "tc/hdl ratio": "TC:HDL ratio", "cholesterol:hdl ratio": "TC:HDL ratio", "tc:hdl ratio": "TC:HDL ratio",
    "body height": "Height", "height": "Height",
    "body weight": "Weight", "weight": "Weight",
    "body mass index (bmi) [ratio]": "BMI", "bmi": "BMI",
    "blood pressure": "Blood pressure",
}

# canonical item -> (target unit, {source unit: factor to target})
UNIT_FACTORS = {
    "Triglycerides":       ("mmol/L", {"mg/dl": 0.01129, "mmol/l": 1.0}),
    "Total cholesterol":   ("mmol/L", {"mg/dl": 0.02586, "mmol/l": 1.0}),
    "HDL cholesterol":     ("mmol/L", {"mg/dl": 0.02586, "mmol/l": 1.0}),
    "LDL cholesterol":     ("mmol/L", {"mg/dl": 0.02586, "mmol/l": 1.0}),
    "Non-HDL cholesterol": ("mmol/L", {"mg/dl": 0.02586, "mmol/l": 1.0}),
}

def parse_tis_datetime(series):
    def parse_one(x):
        if pd.isna(x) or str(x).strip() == "":
            return pd.NaT
        digits = str(x).split(".")[0].zfill(14)  # pandas int inference can drop day's leading zero
        return datetime.strptime(digits, "%d%m%Y%H%M%S")
    return series.apply(parse_one)

In [ ]:
def normalize_long(df, origin):
    out = pd.DataFrame({
        "timestamp": parse_tis_datetime(df["Test or record date (UTC)"]),
        "From": df["From"],
        "canonical_item": df["Test or item"].str.strip().str.lower().map(CONCEPT_MAP).fillna(df["Test or item"]),
        "value_raw": pd.to_numeric(df["Result as recorded"], errors="coerce"),
        "unit_raw": df["Unit"].str.strip().str.lower(),
    })
    out["origin"] = origin

    def to_std(row):
        target, factors = UNIT_FACTORS.get(row["canonical_item"], (row["unit_raw"], {row["unit_raw"]: 1.0}))
        factor = factors.get(row["unit_raw"], 1.0)
        return pd.Series({"value_std": row["value_raw"] * factor, "unit_std": target})

    out[["value_std", "unit_std"]] = out.apply(to_std, axis=1)
    out = out.drop(columns=["value_raw", "unit_raw"])

    # "146/76" isn't numeric - split into its own systolic/diastolic rows
    bp_mask = out["canonical_item"] == "Blood pressure"
    if bp_mask.any():
        bp = df.loc[bp_mask, "Result as recorded"].str.split("/", expand=True).astype(float)
        systolic = out.loc[bp_mask].assign(canonical_item="Systolic BP", value_std=bp[0], unit_std="mmHg")
        diastolic = out.loc[bp_mask].assign(canonical_item="Diastolic BP", value_std=bp[1], unit_std="mmHg")
        out = pd.concat([out[~bp_mask], systolic, diastolic], ignore_index=True)

    return out


# wearable_stream.csv is wide (one row/day); reshape to the same long schema.
# Factors reverse-engineered by matching rows against the TIS bundle's "Wearable reading" entries.
WEARABLE_METRICS = {
    "recovery_score":     ("Recovery score", 1, "score"),
    "resting_heart_rate": ("Resting heart rate", 1, "bpm"),
    "hrv_rmssd":          ("VHR (RMSSD)", 1000, "ms"),           # seconds -> ms
    "quality_duration":   ("Hours asleep", 1 / 3_600_000, "h"),  # ms -> h
    "time_in_bed":        ("Time in bed", 1 / 3_600_000, "h"),   # ms -> h
}

def normalize_wearable(df):
    ts = pd.to_datetime(df["created_at"]).dt.tz_localize(None)
    rows = [
        pd.DataFrame({
            "timestamp": ts, "From": "Wearable", "canonical_item": canonical_item,
            "value_std": df[col] * factor, "unit_std": unit, "origin": "Individual system",
        })
        for col, (canonical_item, factor, unit) in WEARABLE_METRICS.items()
    ]
    return pd.concat(rows, ignore_index=True)

In [ ]:
normalized = []
for name, df in {**individual, **tis_bundle}.items():
    origin = "TIS bundle" if name in tis_bundle else "Individual system"
    if "created_at" in df.columns:
        normalized.append(normalize_wearable(df))
    elif "Test or record date (UTC)" in df.columns:
        normalized.append(normalize_long(df, origin))
    else:
        print(f"Skipped {name} (unrecognized columns): {list(df.columns)}")

combined = pd.concat(normalized, ignore_index=True).dropna(subset=["timestamp"]).sort_values("timestamp")
combined.head()

## Step 3: comparison plots - fragmented view vs. TIS

For a given parameter, each individual system only ever sees its own slice of history (e.g. NHS only sees the UK visits). The TIS bundle assembles the same points into one continuous timeline. Small multiples, one axis, one panel per source plus a combined panel - the point is the gap, not just the number.

In [ ]:
import matplotlib.pyplot as plt

# Okabe-Ito colorblind-safe set, one fixed color per source (never reassigned by rank)
FROM_COLORS = {"India": "#0072B2", "UK": "#E69F00", "Wearable": "#009E73"}

def plot_fragmentation(canonical_item):
    data = combined[(combined["canonical_item"] == canonical_item) & combined["value_std"].notna()].sort_values("timestamp")
    if data.empty:
        print(f"No numeric data for {canonical_item}")
        return

    sources = [s for s in FROM_COLORS if s in data["From"].unique()]
    unit = data["unit_std"].iloc[0]
    fig, axes = plt.subplots(1, len(sources) + 1, figsize=(4.5 * (len(sources) + 1), 3.5), sharey=True)

    for ax, src in zip(axes, sources):
        d = data[(data["origin"] == "Individual system") & (data["From"] == src)]
        ax.plot(d["timestamp"], d["value_std"], "o-", color=FROM_COLORS[src], linewidth=2, markersize=6)
        ax.set_title(f"{src} system alone\n({len(d)} points)")
        ax.tick_params(axis="x", rotation=45)

    ax = axes[-1]
    ax.plot(data["timestamp"], data["value_std"], "-", color="#999999", linewidth=1, zorder=1)
    for src in sources:
        d = data[data["From"] == src]
        ax.scatter(d["timestamp"], d["value_std"], color=FROM_COLORS[src], label=src, zorder=2, s=40)
    ax.set_title(f"TIS: all sources\n({len(data)} points)")
    ax.legend(fontsize=8)
    ax.tick_params(axis="x", rotation=45)

    axes[0].set_ylabel(unit)
    fig.suptitle(f"{canonical_item}: fragmented per-system view vs. the TIS bundle", y=1.05)
    fig.tight_layout()
    plt.show()

for item in ["Total cholesterol", "Systolic BP", "LDL cholesterol", "Resting heart rate", "Hours asleep", "Time in bed"]:
    plot_fragmentation(item)